# Solar Prediction Colab Smoke Check (Hugging Face)

Use this notebook in a fresh Colab runtime to verify the repository end-to-end without Google Drive.
It clones the GitHub repository, optionally pulls data from a Hugging Face Dataset repo (or uses the tracked sample CSV), runs a 1-epoch comparison smoke test, and optionally uploads the resulting artifacts to a Hugging Face Model repo.

To push artifacts to Hugging Face Hub, add a `HF_TOKEN` secret in Colab (or set `os.environ['HF_TOKEN']`).

In [ ]:
GIT_REPO_URL = 'https://github.com/itxtx/solar_prediction.git'
PROJECT_DIR = '/content/solar_prediction'

# Hugging Face Hub configuration
HF_DATASET_REPO = 'itxtx/solar-prediction-data'
HF_DATASET_FILENAME = None  # Set to 'solar_weather.csv' to test HF dataset download, or None to use tracked sample
HF_MODEL_REPO = 'itxtx/solar-prediction-artifacts'
HF_PATH_IN_REPO = 'colab_smoke_check'
UPLOAD_TO_HF = False  # Set True to verify artifact upload to HF_MODEL_REPO

OUTPUT_DIR = 'artifacts/colab_smoke_check'


In [ ]:
import os
import subprocess
from pathlib import Path

if not (Path.cwd() / 'pyproject.toml').exists():
    project = Path(PROJECT_DIR)
    if not project.exists():
        subprocess.run(['git', 'clone', GIT_REPO_URL, str(project)], check=True)
    os.chdir(project)

print('Working directory:', Path.cwd())


In [ ]:
%pip install -q huggingface_hub
%pip install -q --no-deps -e .


In [ ]:
import pandas as pd
from solar_prediction.hf_hub import download_dataset_file, upload_artifacts_folder

if HF_DATASET_FILENAME:
    sample_path = download_dataset_file(HF_DATASET_REPO, HF_DATASET_FILENAME, local_dir='data')
else:
    sample_path = Path('data/sample/SolarPrediction_sample.csv')

assert sample_path.exists(), f'Dataset not found: {sample_path}'
df = pd.read_csv(sample_path)
print('Dataset:', sample_path, 'Shape:', df.shape)
df.head()


In [ ]:
!solar-predict compare --data {sample_path} --epochs 1 --hidden-dim 8 --batch-size 64 --output {OUTPUT_DIR} --quiet


In [ ]:
if UPLOAD_TO_HF:
    commit_url = upload_artifacts_folder(
        OUTPUT_DIR,
        HF_MODEL_REPO,
        path_in_repo=HF_PATH_IN_REPO,
        commit_message='Upload smoke check artifacts',
    )
    print('Uploaded artifacts to Hugging Face:', commit_url)
else:
    print('Skipping Hugging Face upload (UPLOAD_TO_HF=False). Local artifacts in:', OUTPUT_DIR)


Expected result: a CSV-style metrics table with `persistence`, `seasonal_naive`, `lstm`, and `gru`. The neural numbers from one epoch are just a smoke test.